# 報酬で「歩き方の性格」をデザインする四足ロボット

四足ロボット **Unitree Go1** に強化学習（PPO）で歩き方を学習させ、**報酬の重みだけ**を変えて
「標準」「配膳モード」「省エネモード」「元気モード」の歩き方を作り分けます。
シミュレーターは [MuJoCo Playground](https://github.com/google-deepmind/mujoco_playground) です。

| 性格 | 報酬の変え方 | 想定する用途 |
|---|---|---|
| 標準 | 既定のまま | 比較の基準 |
| 配膳モード | 胴体の傾き・揺れへの罰を強化 | 背中に物を載せて運ぶ |
| 省エネモード | エネルギー・急な動きへの罰を強化、足を低く | バッテリーを長持ちさせる |
| 元気モード | 足を高く、滞空時間を長く | 段差や凸凹道 |

**このノートブックの流れ**

1. 標準の歩き方を学習する
2. その結果を出発点に、報酬の重みを変えて各性格を追加学習（ファインチューニング）する
3. 全員に同じ指令（静止 → 前進 → 速歩 → 旋回 → 横歩き）を出して評価する
4. 図・比較動画・提出用の動画（約1分40秒）を自動で作る

**実行する前に**

- メニュー「ランタイム」→「ランタイムのタイプを変更」で **GPU（T4）** を選ぶ（Colab Pro なら L4 / A100）
- 上から順に ▶ で実行する。途中で切断されても、**同じセルをもう一度実行すれば続きから再開**します（結果は Google ドライブに保存されます）
- 最初は `PROFILE` を **trial（動作確認）** にして最後まで通し、問題がなければ本番の設定で実行し直すのがおすすめです

In [ ]:
#@title 0. 設定（最初に1回だけ編集）
REPO_URL = "https://github.com/<GitHubユーザー名>/go1-gait-personality.git"  #@param {type:"string"}
DRIVE_DIR = "/content/drive/MyDrive/go1-gait-personality"  #@param {type:"string"}
PROFILE = "trial（動作確認・A100で約35分）"  #@param ["trial（動作確認・A100で約35分）", "t4（T4向け・数時間）", "pro（A100で約2時間）"]
IMPL = "default"  #@param ["default", "jax"]
STYLES = "serving,eco,energetic"  #@param {type:"string"}
EVAL_SEEDS = 8  #@param {type:"integer"}

# PROFILE ごとの学習量（環境ステップ数）
#   trial: 標準 1,000万 + 各性格 500万   … 動作確認用。歩き方はまだ不完全
#   t4   : 標準 1億     + 各性格 4,000万 … 無料版 T4 向け
#   pro  : 標準 2億     + 各性格 6,000万 … L4 / A100 向け（公式と同じ学習量）
#   ※ Brax が約1,470万ステップ単位に切り上げるので、実際は少し多くなります
# IMPL: 物理エンジンの計算方式。エラーが出たときだけ "jax" に変更
# 所要時間は目安です。実行中に残り時間が表示されます。
import json
json.dump(dict(REPO_URL=REPO_URL, DRIVE_DIR=DRIVE_DIR, PROFILE=PROFILE, IMPL=IMPL,
               STYLES=STYLES, EVAL_SEEDS=EVAL_SEEDS),
          open("/content/gait_settings.json", "w"), ensure_ascii=False)
print("設定を保存しました:", PROFILE)

## 1. 準備（3〜5分）

GPU の確認、コードの取得、ライブラリのインストールを行います。
ライブラリの更新で再起動が必要なときは**自動で再起動**します。その場合は「2.」から続けてください（設定は保存済みです）。

In [ ]:
#@title 1. 準備：GPU 確認・コード取得・インストール
import json, os, subprocess, sys

S = json.load(open("/content/gait_settings.json"))
if subprocess.run(["nvidia-smi"], capture_output=True).returncode != 0:
    raise RuntimeError("GPU が見つかりません。「ランタイム」→「ランタイムのタイプを変更」で GPU を選んでください。")
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout)

if "<" in S["REPO_URL"]:
    raise ValueError("0. 設定の REPO_URL を、自分の GitHub リポジトリの URL に書き換えてください。")
REPO_DIR = "/content/go1-gait-personality"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "-q", S["REPO_URL"], REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "-q"], check=True)

# 動画の日本語字幕用フォント
subprocess.run("apt-get -qq install -y fonts-noto-cjk > /dev/null 2>&1", shell=True)

import numpy  # インストールで numpy が更新されたかを調べるため
np_before = numpy.__version__
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                f"{REPO_DIR}/requirements-colab.txt"], check=True)
np_after = subprocess.run([sys.executable, "-c", "import numpy; print(numpy.__version__)"],
                          capture_output=True, text=True).stdout.strip()

# 画面なしで描画するための EGL 設定（公式 MuJoCo Playground ノートブックと同じ）
ICD = "/usr/share/glvnd/egl_vendor.d/10_nvidia.json"
if not os.path.exists(ICD):
    os.makedirs(os.path.dirname(ICD), exist_ok=True)
    with open(ICD, "w") as f:
        f.write('{"file_format_version": "1.0.0", "ICD": {"library_path": "libEGL_nvidia.so.0"}}')

if np_before != np_after:
    print(f"numpy が {np_before} → {np_after} に更新されたため、ランタイムを再起動します。")
    print("再起動後は「2.」のセルから実行してください。")
    os.kill(os.getpid(), 9)
print("✅ 準備完了")

## 2. Google ドライブの接続と読み込み

学習結果はすべて Google ドライブに保存されます。初回は接続の許可を求められます。

In [ ]:
#@title 2. Google ドライブの接続・読み込み
import json, os, sys
os.environ["MUJOCO_GL"] = "egl"
os.environ["XLA_FLAGS"] = os.environ.get("XLA_FLAGS", "") + " --xla_gpu_triton_gemm_any=True"

from google.colab import drive
drive.mount("/content/drive")

S = json.load(open("/content/gait_settings.json"))
sys.path.insert(0, "/content/go1-gait-personality/src")

import jax
print("JAX devices:", jax.devices())
assert jax.devices()[0].platform == "gpu", "GPU が使われていません。「1.」をやり直してください。"

from IPython.display import HTML, Image as IPImage, Markdown, display
from gait_personality import describe, plots
from gait_personality.pipeline import (RunConfig, evaluate, finetune_styles, histories,
                                       load_results, make_figures, make_videos, train_base)

PROFILES = {
    "trial": dict(base_timesteps=10_000_000, finetune_timesteps=5_000_000),
    "t4": dict(base_timesteps=100_000_000, finetune_timesteps=40_000_000),
    "pro": dict(base_timesteps=200_000_000, finetune_timesteps=60_000_000),
}
profile = S["PROFILE"].split("（")[0]
cfg = RunConfig(
    run_dir=f"{S['DRIVE_DIR']}/runs/{profile}",
    styles=tuple(s.strip() for s in S["STYLES"].split(",") if s.strip()),
    impl=None if S["IMPL"] == "default" else S["IMPL"],
    eval_seeds=int(S["EVAL_SEEDS"]),
    **PROFILES[profile],
)
print("保存先:", cfg.run_dir, "\n")
for s in cfg.all_styles:
    print(describe(s), "\n")

## 3. 標準の歩き方を学習（いちばん時間がかかります）

最初の数分はプログラムの準備（コンパイル）で、表示が止まって見えますが正常です。
評価のたびに進み具合と残り時間の目安が表示されます。**ブラウザのタブは開いたまま**にしてください。

In [ ]:
#@title 3. 標準の歩き方を学習
hist = train_base(cfg)
display(plots.training_curves({"standard": hist}))

## 4. 性格ごとの追加学習

標準の歩き方を出発点に、報酬の重みだけを変えて追加学習します。
重みの値は `src/gait_personality/styles.py` の `STYLES` で定義しています。

In [ ]:
#@title 4. 性格ごとの追加学習
finetune_styles(cfg)
display(plots.training_curves(histories(cfg)))

## 5. 評価：全員に同じ指令を出して比べる

静止（2秒）→ 前進 0.6 m/s（5秒）→ 前進 1.2 m/s（5秒）→ 前進しながら左旋回（4秒）→ 左へ横歩き（4秒）を、
性格ごとに `EVAL_SEEDS` 回ずつ実行し、指標を計算します。

| 指標 | 意味 |
|---|---|
| 移動コスト（Cost of transport） | 1 m 進むのに使った関節の仕事 ÷（体重 × 重力加速度）。小さいほど省エネ |
| 速度の追従誤差 | 指令した速度と実際の速度のずれ（RMS）。小さいほど正確 |
| 胴体の揺れ | 胴体の回転速度（ロール・ピッチ）の RMS。小さいほど安定 |
| 胴体の上下動 | 胴体の高さの標準偏差 |
| 足の高さ | 1歩ごとの足先の最高点の平均 |

In [ ]:
#@title 5. 評価
df, summary, logs = evaluate(cfg)
display(Markdown(cfg.path("results", "summary.md").read_text()))

## 6. 図の作成

In [ ]:
#@title 6. 図の作成
if "summary" not in globals():  # 再起動後に 5. を飛ばした場合は保存済みの結果を読む
    summary, logs, _ = load_results(cfg)
figures = make_figures(cfg, summary)
for path in figures.values():
    display(IPImage(filename=str(path), width=900))

## 7. 動画の作成

- `videos/final_video.mp4`：提出用（約1分40秒、等速）。タイトル → 各性格の歩き方 → 4体の比較 → 結果の図 → まとめ
- `videos/comparison_grid.mp4`：4体を並べた比較だけの動画（20秒、README 用）

In [ ]:
#@title 7. 動画の作成（数分）
from base64 import b64encode
if "logs" not in globals() or "figures" not in globals():
    summary, logs, figures = load_results(cfg)
videos = make_videos(cfg, summary, logs, figures)

def show_mp4(path, width=800):
    data = b64encode(open(path, "rb").read()).decode()
    display(HTML(f'<video width="{width}" controls src="data:video/mp4;base64,{data}"></video>'))

show_mp4(str(videos["grid"]))
print("提出用の動画:", videos["final"])

## 8. 提出物への反映

結果は Google ドライブの `go1-gait-personality/runs/<PROFILE>/` にあります。

| ファイル | 使い道 |
|---|---|
| `videos/final_video.mp4` | YouTube に「限定公開」以上でアップロード（タイトル：`PAI最終課題_omnicampusアカウント名`） |
| `results/summary.md`, `summary.csv`, `*.png` | ローカルのリポジトリの `results/` にコピーして commit・push。レポートと README で使う |
| `checkpoints/`, `eval/` | 大きいので GitHub には入れない（`.gitignore` 済み） |

動画・レポート・コードには氏名やメールアドレスなどの個人情報を入れないでください。

## 困ったとき

| 症状 | 対処 |
|---|---|
| `GPU が見つかりません` | 「ランタイムのタイプを変更」で GPU を選ぶ |
| 途中で切断された | 「1.」→「2.」→ 中断したセル、の順に実行。学習は続きから再開します |
| GitHub のコードを更新した | 「ランタイム」→「セッションを再起動」してから上から実行（読み込み済みのコードは再起動しないと更新されません） |
| `RESOURCE_EXHAUSTED`（GPU メモリ不足） | 自動で並列数を半分にして再試行します |
| MuJoCo Warp に関するエラー | 「0. 設定」で `IMPL` を `jax` にして「2.」から実行し直す |
| 動画の日本語が □ になる | 「1.」をもう一度実行（日本語フォントの導入） |
| すぐ転ぶ・歩かない | `PROFILE` を増やす、または `styles.py` の重みを既定値に近づける |